<a href="https://colab.research.google.com/github/yo-danny/speech-emotion-recognition/blob/main/speech_emotion_recognition_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Speech Emotion Recognition



In [1]:
import librosa
import soundfile
import os, glob, pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from librosa.feature import spectral_centroid, spectral_rolloff, zero_crossing_rate

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

from keras import Sequential
from keras.layers import Dense, LSTM, Dropout, Activation
from keras.layers import MaxPooling1D, Conv1D
from keras.utils import np_utils
from keras.callbacks import ModelCheckpoint

Defining a function to extract mfcc, chroma and mel features from the audio.

Being:

- mfcc: Mel Frequency Cepstral Coefficient, represents the short-term power spectrum of a sound
- chroma: Pertains to the 12 different pitch classes
- mel: Mel Spectrogram Frequency

In [9]:
def extract_feature(file_name, mfcc, chroma, mel, spectral_centroid, spectral_rolloff, zero_crossing_rate):
    with soundfile.SoundFile(file_name) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate=sound_file.samplerate
        if chroma:
            stft=np.abs(librosa.stft(X))
        result=np.array([])
        if mfcc: # mapping sound to human perception to identify phonetic content
            mfccs=np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
            result=np.hstack((result, mfccs))
        if chroma: # analyzing the harmonic and tonal content
            chroma=np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T,axis=0)
            result=np.hstack((result, chroma))
        if mel: # visual representations of audio frequency over time
            mel=np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T,axis=0)
            result=np.hstack((result, mel))
        if spectral_centroid: # tells you where the middle of the frequency spectrum is (the "average" frequency)
            spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=X, sr=sample_rate).T,axis=0)
            result=np.hstack((result, spectral_centroid))
        if spectral_rolloff: # tells you where the edge of the majority of energy lies (usually 95% or 85% of the energy is below this point)
            spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=X, sr=sample_rate).T,axis=0)
            result=np.hstack((result, spectral_rolloff))
        if zero_crossing_rate: # counts how often the signal crosses the zero axis, used to distinguish noisy, high-energy emotions
            zero_crossing_rate = np.mean(librosa.feature.zero_crossing_rate(y=X).T,axis=0)
            result=np.hstack((result, zero_crossing_rate))
    return result

Dictionary to hold number values for emotions avaible in the RAVDESS dataset.

In [3]:
emotions = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

Function to load the data from our ambient, calls the extraction of features and return the train and test datasets for model training.

In [4]:
def load_data(test_size=0.2):
    x, y = [], []
    for file in glob.glob("/content/drive/MyDrive/speech-emotion-recognition-ravdess-data/Actor_*/*.wav"):
        file_name = os.path.basename(file)
        emotions = file_name.split("-")[2]
        feature = extract_feature(file_name=file, mfcc=True, chroma=True, mel=True, spectral_centroid=True, spectral_rolloff=True, zero_crossing_rate=True)
        x.append(feature)
        y.append(emotions)
    return np.array(np.expand_dims(x, axis=2)), np.array(y)


In [5]:
x_train, x_test, y_train, y_test = load_data(test_size=0.25)

lb = LabelBinarizer()
y_train_encoded = lb.fit_transform(y_train)
y_test_encoded = lb.transform(y_test)

Initializing the CNN model.

In [ ]:
model = Sequential()
model.add(Conv1D(64, 5, padding='same', input_shape=(x_train.shape[1], 1)))
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=8))
model.add(LSTM(128))
model.add(Dense(8))
model.add(Activation('softmax'))
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train_encoded, batch_size = 16, epochs=50, validation_data=(x_test, y_test_encoded))

Predicting and evaluating the model.

In [8]:
y_pred = model.predict(x_test)

y_pred_encoded = lb.inverse_transform(y_pred)

accuracy=accuracy_score(y_true=y_test, y_pred=y_pred_encoded)
precision=precision_score(y_true=y_test, y_pred=y_pred_encoded, average='macro')
recall=recall_score(y_true=y_test, y_pred=y_pred_encoded, average='macro')
f1=f1_score(y_true=y_test, y_pred=y_pred_encoded, average='macro')

print(model.summary())

print("Accuracy: {:.2f}%".format(accuracy*100))
print("Precision: {:.2f}%".format(precision*100))
print("Recall: {:.2f}%".format(recall*100))
print("F1-Score: {:.2f}%".format(f1*100))

sns.heatmap(confusion_matrix(y_test, y_pred_encoded), annot=True)

Accuracy: 45.88%
